# Joint, Conditional, and Marginal Distributions

| Difficulty | ████░░░░░░ 4/10 |
| :--- | :--- |
| Prerequisites | 03.03 |
| Time | ~65 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/03_Probability_and_Statistics/04_joint_conditional_marginal.ipynb)

---

## 🎯 Learning Objective

Work with multiple random variables simultaneously through joint distributions, and learn to compute marginals and conditionals, and test for independence.

---

## 📐 Theory

Real models never deal with a single random variable in isolation — an image has millions of
correlated pixels, a sentence is a sequence of correlated tokens. **Joint distributions**
extend everything from `03.02`-`03.03` to multiple random variables at once.

### Joint distributions

For two discrete random variables, the **joint PMF** is $p_{X,Y}(x,y) = P(X=x, Y=y)$, still
governed by the same axioms: $p_{X,Y}(x,y)\ge 0$ and $\sum_x\sum_y p_{X,Y}(x,y)=1$. For
continuous variables, the **joint PDF** $f_{X,Y}(x,y)$ satisfies $P((X,Y)\in A) =
\iint_A f_{X,Y}(x,y)\,dx\,dy$ — the double-integral machinery of `02.08`
(link: [Multivariable Calculus](../02_Calculus/08_multivariable_calculus.ipynb)) computing
probability instead of volume.

### Marginals: integrating a variable out

The **marginal distribution** of $X$ recovers $X$'s own distribution by summing/integrating
$Y$ away entirely:

$$p_X(x) = \sum_y p_{X,Y}(x,y), \qquad f_X(x) = \int_{-\infty}^{\infty} f_{X,Y}(x,y)\,dy.$$

This is literally "what happens to $X$ if we stop tracking $Y$" — every joint distribution
determines its marginals uniquely, but (critically) the reverse is false: the marginals alone
don't determine the joint, since they say nothing about how $X$ and $Y$ relate.

### Conditional distributions

The **conditional PMF/PDF** of $Y$ given $X=x$ restricts to a slice of the joint and
renormalizes, generalizing `03.01`'s conditional probability to whole distributions:

$$p_{Y\mid X}(y\mid x) = \frac{p_{X,Y}(x,y)}{p_X(x)}, \qquad f_{Y\mid X}(y\mid x) = \frac{f_{X,Y}(x,y)}{f_X(x)}.$$

Fixing $x$ and varying $y$ traces out a full, valid one-variable distribution (it still sums/
integrates to 1 over $y$) — conditioning collapses a joint distribution down to a slice, the
same "restrict and renormalize" idea as before, just applied to a whole density instead of a
single event.

### Independence of random variables

$X$ and $Y$ are **independent** exactly when the joint factors into the product of marginals
for *every* $(x,y)$:

$$p_{X,Y}(x,y) = p_X(x)\,p_Y(y) \quad\text{for all } x,y.$$

Equivalently, $f_{Y\mid X}(y\mid x) = f_Y(y)$ — conditioning on $X$ changes nothing about $Y$'s
distribution. Independence is the strongest possible statement that two variables carry no
information about each other; `03.03`'s "zero covariance" is a much weaker, linear-only
condition that independence implies but does not require (independent variables always have
zero covariance, but zero covariance does not imply independence — two variables can be
uncorrelated yet still deterministically related through a nonlinear function).

### The chain rule for more than two variables

Applying conditional distributions repeatedly **factors any joint distribution** into a product
of conditionals, in any order:

$$p(x_1,\ldots,x_n) = p(x_1)\,p(x_2\mid x_1)\,p(x_3\mid x_1,x_2)\cdots p(x_n\mid x_1,\ldots,x_{n-1}) = \prod_{i=1}^n p(x_i \mid x_{<i}).$$

This identity — first seen as the multiplication rule for events in `03.01` — is exactly how
autoregressive models factor a joint distribution over an entire sequence into next-token
conditionals, demonstrated below.

---

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import multivariate_normal, norm
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

A correlated joint density, its two marginals (obtained by integrating the heatmap along one
axis), and a conditional slice (a single column of the heatmap, renormalized) — three views of
the same joint distribution.

In [ ]:
# A correlated bivariate Gaussian: joint density, both marginals (via numerical integration),
# and one conditional slice -- three views of the SAME joint distribution.
rho = 0.7
mean = [0.0, 0.0]
cov = [[1.0, rho], [rho, 1.0]]
joint_rv = multivariate_normal(mean, cov)

grid_1d = np.linspace(-3.5, 3.5, 300)
Xg, Yg = np.meshgrid(grid_1d, grid_1d)
joint_density = joint_rv.pdf(np.dstack((Xg, Yg)))

fig = plt.figure(figsize=(12, 5.5))
gs = fig.add_gridspec(2, 3, height_ratios=[1, 3], width_ratios=[3, 1, 1.3], hspace=0.05, wspace=0.35)

ax_joint = fig.add_subplot(gs[1, 0])
ax_joint.contourf(Xg, Yg, joint_density, levels=20, cmap="viridis")
x_slice = 1.5
ax_joint.axvline(x_slice, color="white", ls="--", lw=2)
ax_joint.set_xlabel("x"); ax_joint.set_ylabel("y")
ax_joint.set_title(f"Joint density f(x,y)  [rho={rho}]\nwhite line: slice at x={x_slice}")

ax_top = fig.add_subplot(gs[0, 0], sharex=ax_joint)
marginal_x_numeric = joint_density.sum(axis=0) * (grid_1d[1] - grid_1d[0])  # integrate out y (rows)
ax_top.plot(grid_1d, marginal_x_numeric, color="#4C72B0")
ax_top.plot(grid_1d, norm.pdf(grid_1d), "k--", lw=1, label="true marginal N(0,1)")
ax_top.set_title("Marginal f(x) = integral over y"); ax_top.legend(fontsize=7); ax_top.set_yticks([])

ax_right = fig.add_subplot(gs[1, 1], sharey=ax_joint)
marginal_y_numeric = joint_density.sum(axis=1) * (grid_1d[1] - grid_1d[0])  # integrate out x (cols)
ax_right.plot(marginal_y_numeric, grid_1d, color="#4C72B0")
ax_right.plot(norm.pdf(grid_1d), grid_1d, "k--", lw=1)
ax_right.set_title("Marginal\nf(y)", fontsize=9); ax_right.set_xticks([])

ax_cond = fig.add_subplot(gs[1, 2])
col_idx = np.argmin(np.abs(grid_1d - x_slice))
joint_slice = joint_density[:, col_idx]
conditional_y_given_x = joint_slice / (joint_slice.sum() * (grid_1d[1] - grid_1d[0]))  # renormalize the slice
true_conditional = norm(loc=rho * x_slice, scale=np.sqrt(1 - rho**2))  # exact bivariate-Gaussian formula
ax_cond.plot(conditional_y_given_x, grid_1d, color="#C44E52", label="from joint slice")
ax_cond.plot(true_conditional.pdf(grid_1d), grid_1d, "k--", lw=1, label="exact formula")
ax_cond.set_title(f"Conditional\nf(y|x={x_slice})", fontsize=9); ax_cond.legend(fontsize=6); ax_cond.set_xticks([])

plt.show()


def trapz_1d(values, x):
    # Hand-rolled trapezoidal rule -- avoids relying on np.trapz (removed in NumPy 2.0+)
    # or np.trapezoid (absent before it), since either could be missing depending on version.
    widths = np.diff(x)
    return np.sum(widths * (values[:-1] + values[1:]) / 2)


print(f"Marginal f(x) integrates to: {trapz_1d(marginal_x_numeric, grid_1d):.4f} (expect 1)")
print(f"Conditional f(y|x={x_slice}) integrates to: {trapz_1d(conditional_y_given_x, grid_1d):.4f} (expect 1)")
print(f"Conditional mean E[Y|X={x_slice}] from slice: {trapz_1d(grid_1d * conditional_y_given_x, grid_1d):.4f} "
      f"(exact formula rho*x = {rho * x_slice:.4f})")

## 🐍 Implementation from Scratch

We build a discrete joint PMF as an explicit table, then compute marginals, conditionals, and
an independence test directly from the table — no shortcuts, just the sum/normalize
definitions from the Theory section applied to actual numbers.

In [ ]:
# Joint PMF as an explicit table: X = "weather" (0=sunny,1=rainy), Y = "commute" (0=bike,1=car,2=bus)
# Built so that weather and commute choice are NOT independent (rain shifts commute mode)
joint_pmf = np.array([
    # bike   car    bus     <- Y
    [0.30, 0.10, 0.10],   # X=sunny
    [0.05, 0.25, 0.20],   # X=rainy
])
assert np.isclose(joint_pmf.sum(), 1.0)
x_labels, y_labels = ["sunny", "rainy"], ["bike", "car", "bus"]

# Marginals: sum the joint table along the OTHER axis
marginal_X = joint_pmf.sum(axis=1)   # sum over y (columns) -> P(X=x)
marginal_Y = joint_pmf.sum(axis=0)   # sum over x (rows)    -> P(Y=y)
print("Joint PMF table:\n", joint_pmf)
print(f"\nMarginal P(X) [{x_labels}]: {marginal_X}  (sums to {marginal_X.sum():.4f})")
print(f"Marginal P(Y) [{y_labels}]: {marginal_Y}  (sums to {marginal_Y.sum():.4f})")

# Conditional P(Y | X=rainy): take the rainy ROW and renormalize by P(X=rainy)
row_rainy = joint_pmf[1, :]
conditional_Y_given_rainy = row_rainy / marginal_X[1]
print(f"\nP(Y | X=rainy) = {np.round(conditional_Y_given_rainy, 4)}  (sums to {conditional_Y_given_rainy.sum():.4f})")
print(f"Compare P(Y) unconditional = {np.round(marginal_Y, 4)}  -- rain clearly shifts commute mode")

# Independence test: is joint_pmf[i,j] == marginal_X[i] * marginal_Y[j] for EVERY cell?
outer_product_if_independent = np.outer(marginal_X, marginal_Y)
print(f"\nIf X,Y were independent, joint would be:\n{np.round(outer_product_if_independent, 4)}")
print(f"Actual joint:\n{joint_pmf}")
print(f"Independent? {np.allclose(joint_pmf, outer_product_if_independent)}  "
      f"(False -- weather and commute mode are dependent, as built)")

# Chain rule for THREE variables: draw a bigram-style sequence by chaining conditionals,
# P(x1,x2,x3) = P(x1) P(x2|x1) P(x3|x1,x2) -- verify the chained product matches direct enumeration
rng = np.random.default_rng(0)
vocab = ["A", "B"]
p_x1 = np.array([0.6, 0.4])
p_x2_given_x1 = np.array([[0.7, 0.3], [0.2, 0.8]])           # rows indexed by x1
p_x3_given_x1x2 = np.array([[[0.9, 0.1], [0.4, 0.6]],        # indexed by [x1][x2]
                             [[0.5, 0.5], [0.1, 0.9]]])

chained_prob_ABA = p_x1[0] * p_x2_given_x1[0, 1] * p_x3_given_x1x2[0, 1, 0]  # P(A,B,A)
print(f"\nChain rule: P(x1=A, x2=B, x3=A) = P(A)*P(B|A)*P(A|A,B) = {chained_prob_ABA:.6f}")

## 🤖 Why This Matters for AI

A **multi-output model** (e.g. predicting both a bounding box and a class label) predicts a
joint distribution over its outputs, and whether it should model their full correlation or
treat them as conditionally independent given the input is a real architectural decision.
**Autoregressive language models** go further: they never model the joint
$P(\text{token}_1,\ldots,\text{token}_L)$ directly (impossible — it has $V^L$ entries for
vocabulary size $V$), they factor it into a product of conditionals
$\prod_t P(\text{token}_t \mid \text{token}_{<t})$, and a transformer's entire job is to compute
each conditional term in that product.

---

In [ ]:
# A tiny autoregressive "language model": a bigram transition table IS a set of conditional
# distributions P(token_t | token_{t-1}) -- generating text = repeatedly sampling from them.
rng = np.random.default_rng(0)
vocab = ["the", "cat", "sat", "ran", "."]
V = len(vocab)

# transition[i, j] = P(next=vocab[j] | current=vocab[i]) -- each ROW is a valid conditional PMF
transition = np.array([
    [0.05, 0.55, 0.05, 0.35, 0.00],   # after "the": likely "cat" or "ran"
    [0.05, 0.05, 0.60, 0.05, 0.25],   # after "cat": likely "sat"
    [0.10, 0.05, 0.05, 0.05, 0.75],   # after "sat": likely "."
    [0.08, 0.04, 0.04, 0.04, 0.80],   # after "ran": likely "."
    [0.60, 0.20, 0.10, 0.10, 0.00],   # after ".": likely "the" (new sentence)
])
assert np.allclose(transition.sum(axis=1), 1.0), "each row must be a valid conditional PMF"

def generate_sequence(start_idx, length, rng):
    seq = [start_idx]
    for _ in range(length - 1):
        current = seq[-1]
        next_idx = rng.choice(V, p=transition[current])   # sample from P(next | current) directly
        seq.append(next_idx)
    return [vocab[i] for i in seq]

print("Sampled sequences from the factored joint P(seq) = P(w1) * prod_t P(w_t | w_{t-1}):")
for _ in range(4):
    print("  " + " ".join(generate_sequence(start_idx=0, length=6, rng=rng)))

# Compute the joint probability of ONE specific sequence via the chain-rule factorization,
# and cross-check it against brute-force enumeration over the same fixed starting token
seq_indices = [0, 1, 2, 4]  # "the cat sat ."
joint_prob_via_chain_rule = 1.0
for t in range(1, len(seq_indices)):
    joint_prob_via_chain_rule *= transition[seq_indices[t - 1], seq_indices[t]]
print(f"\nP('the cat sat .' | starts with 'the') via chain rule = {joint_prob_via_chain_rule:.6f}")

# Brute-force: enumerate EVERY length-4 sequence starting with 'the' and sum probabilities of
# ones that equal our target, as an independent check that chaining conditionals is correct
from itertools import product as iproduct
total_prob_mass = 0.0
target_prob_mass = 0.0
for cont in iproduct(range(V), repeat=3):
    full = (0,) + cont
    p = 1.0
    for t in range(1, len(full)):
        p *= transition[full[t - 1], full[t]]
    total_prob_mass += p
    if full == tuple(seq_indices):
        target_prob_mass = p
print(f"Brute-force enumeration gives the same sequence probability: {target_prob_mass:.6f}")
print(f"Total probability mass over ALL length-4 continuations of 'the': {total_prob_mass:.6f} (expect 1.0)")

## 🏋️ Exercises

### Warm-up
1. Using the `joint_pmf` table below, verify by hand that the marginal $p_X(x)$ (row sums)
   and $p_Y(y)$ (column sums) each sum to 1. Then check whether $X$ and $Y$ are independent by
   testing $p_{X,Y}(x,y) = p_X(x)p_Y(y)$ for every cell.

### Practice
2. Build a joint PMF table for two dice, $X=$ first die, $Y=\max(X, \text{second die})$.
   Compute the marginal of $Y$ and the conditional $P(Y\mid X=3)$. Confirm $X$ and $Y$ are
   *not* independent (this should be intuitive: knowing the first die constrains what $\max$
   can be).

### Challenge
3. In the autoregressive factorization cell, change the toy vocabulary/transition table so
   that token 2 can *never* follow token 0 (set that conditional probability to zero and
   renormalize the rest of that row). Regenerate 5 sequences and confirm token 2 never
   immediately follows token 0. Then explain why a factored autoregressive model needs only
   $O(V^2)$ numbers (bigram table) or $O(V\cdot d)$ (neural network) to describe a joint
   distribution over sequences that naively would need $O(V^L)$ numbers for length $L$.

---

## 📚 Further Reading
- Wasserman, *All of Statistics*, Ch. 2.4–2.5 (Joint Distributions) and Ch. 3 (Independence)
- Bishop, *Pattern Recognition and Machine Learning*, Ch. 1.2 (Probability Theory, esp. the sum and product rules)
- Jurafsky & Martin, [*Speech and Language Processing*](https://web.stanford.edu/~jurafsky/slp3/), Ch. 3 (N-gram Language Models)

---

*Next notebook: [Bayes' Theorem and Inference](05_bayes_theorem_and_inference.ipynb)*